# Lab 5: Generative Modeling

Two ways to generate an image, one lab. You write three routines they turn on:
the positive beta-VAE training loss with its analytic KL, the forward-diffusion step
that noises an image, and one DDIM step that walks it back. Then you run a controlled
VAE study, inspect reconstructions and fixed-prior samples, and test a small diffusion
model with two valid sampling budgets. Everything runs on FashionMNIST.

**What you will do**

1. Write the positive beta-VAE loss, including its analytic Gaussian KL.
2. Write the forward-diffusion step that sets the denoiser's target.
3. Write one deterministic DDIM reverse step for sampling.
4. Design a VAE to a parameter budget, then change one KL weight.
5. Record a claim, its evidence, a caveat, and one controlled next experiment.

The red **Your Task** banners mark the parts you complete. Everything else is
provided and ready to run.

## At a glance

| | |
|---|---|
| Stack | PyTorch and torchvision; a GPU runtime is recommended |
| You write | `vae_objective`, `q_sample`, `ddim_step`, and an evidence record |
| You submit | Five values, R1 through R5, from the final cell |
| GPU runtime | A few minutes of training in quick mode; longer on the full split |

<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">A VAE and a diffusion model reach the same goal by opposite routes: one compresses an image to a latent and decodes it, the other noises the image and learns to walk it back. Writing the core step of each makes that contrast concrete.</p>
</div>

## What is graded

The four course-page gates separate readiness, the experiment record,
claim-evidence-caveat reasoning, and the next experiment. R1 through R3 check your
three functions on fixed inputs. R4 accepts your VAE design inside the stated budget.
R5 verifies the controlled runs, complete decomposed histories, valid diffusion
schedule, and your completed evidence record. You use measured results to write the
record; the course page does not ask you to guess an exact stochastic metric.

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Sharper reconstructions do not always mean a better model. A tiny KL weight can reconstruct well yet leave gaps in the latent space that turn into nonsense when you sample the prior. Judge the tradeoff, not the reconstructions alone.</p>
</div>

In [ ]:
import math
import random
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.prop_cycle": plt.cycler(
        color=["#A31F34", "#1D4ED8", "#059669", "#B45309", "#7C3AED"]
    ),
})


reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")

In [ ]:
QUICK_MODE = True
DATA_ROOT = "./data"

TRAIN_PER_CLASS = 300
VALIDATION_PER_CLASS = 100 if QUICK_MODE else 300
DEFAULT_BATCH_SIZE = 128
EPOCHS = 3 if QUICK_MODE else 8
NUM_WORKERS = 0 if QUICK_MODE else 2

print(f"quick_mode={QUICK_MODE}, epochs={EPOCHS}")

In [ ]:
transform = transforms.ToTensor()
split_source = datasets.FashionMNIST(DATA_ROOT, train=True, download=True)


def stratified_split(targets, train_per_class, validation_per_class, seed):
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices, validation_indices = [], []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(class_indices[train_per_class:needed].tolist())
    return sorted(train_indices), sorted(validation_indices)


train_indices, validation_indices = stratified_split(
    split_source.targets, TRAIN_PER_CLASS, VALIDATION_PER_CLASS, SEED
)
assert set(train_indices).isdisjoint(validation_indices)

image_source = datasets.FashionMNIST(
    DATA_ROOT, train=True, transform=transform, download=False
)
train_set = Subset(image_source, train_indices)
validation_set = Subset(image_source, validation_indices)


def seeded_loader(dataset, *, batch_size, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", generator=generator,
    )


def trainable_parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("training examples:", len(train_indices),
      "validation examples:", len(validation_indices))

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 1</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>vae_objective</code></div>
</div>

Complete `vae_objective`. Return `(loss, reconstruction, kl)` where the positive
reconstruction term is the mean over the batch of the per-example squared error
summed over pixels, the KL is the mean over the batch of
`-0.5 * sum(1 + logvar - mu^2 - exp(logvar))`, and `loss = reconstruction + beta * kl`.
This minimized quantity is a beta-VAE loss (a negative-ELBO-style surrogate under
the stated reconstruction convention); it is not the ELBO itself.

In [ ]:
def vae_objective(reconstructed, x, mu, logvar, beta):
    """Return the positive beta-VAE loss and its reconstruction and KL terms."""
    # STUDENT TASK 1:
    #   reconstruction = mean over batch of sum over pixels of (reconstructed - x)^2
    #   kl = mean over batch of -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    #   loss = reconstruction + beta * kl
    raise NotImplementedError("Return (loss, reconstruction, kl).")

In [ ]:
kl_probe_mu = torch.tensor([[0.0, 1.0]])
kl_probe_logvar = torch.tensor([[0.0, math.log(4.0)]])
kl_probe_x = torch.zeros(1, 2)
_, kl_probe_reconstruction, kl_probe_kl = vae_objective(
    kl_probe_x, kl_probe_x, kl_probe_mu, kl_probe_logvar, beta=1.0
)
kl_probe_value = round(kl_probe_kl.item(), 5)
kl_probe_contract = (
    abs(kl_probe_reconstruction.item()) < 1e-9
    and abs(kl_probe_value - 1.30685) < 1e-4
)
assert kl_probe_contract, "KL of mu=(0,1), logvar=(0,ln4) is 0.5*(4 - ln4) = 1.30685."
print("Task 1 checks passed. Analytic KL:", kl_probe_value)

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 2</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>q_sample</code></div>
</div>

Complete `q_sample`, the forward process
`x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * noise`. This defines the noisy
inputs whose noise the denoiser learns to predict.

In [ ]:
def q_sample(x_0, alpha_bar, noise):
    """Forward diffusion: x_t = sqrt(alpha_bar) x_0 + sqrt(1 - alpha_bar) noise."""
    # STUDENT TASK 2: scale the clean signal by sqrt(alpha_bar) and the noise by
    # sqrt(1 - alpha_bar), then add them.
    raise NotImplementedError("Return the noised x_t.")

In [ ]:
qsample_alpha_bar = torch.tensor(0.36)
qsample_x0 = torch.tensor([2.0])
qsample_noise = torch.tensor([0.5])
qsample_x_t = q_sample(qsample_x0, qsample_alpha_bar, qsample_noise)
qsample_probe_value = round(qsample_x_t.item(), 4)
qsample_probe_contract = abs(qsample_probe_value - 1.6) < 1e-4
assert qsample_probe_contract, "sqrt(0.36)*2.0 + sqrt(0.64)*0.5 = 0.6*2 + 0.8*0.5 = 1.6."
print("Task 2 checks passed. Noised value x_t:", qsample_probe_value)

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 3</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>ddim_step</code></div>
</div>

Complete `ddim_step`. Estimate the clean signal
`x0_hat = (x_t - sqrt(1 - alpha_bar_t) * predicted_noise) / sqrt(alpha_bar_t)`, then
return the less-noisy state
`sqrt(alpha_bar_prev) * x0_hat + sqrt(1 - alpha_bar_prev) * predicted_noise`.

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Two steps, in order</summary>
<div style="margin-top:8px;">First recover a clean-image estimate <code>x0_hat</code> from <code>x_t</code> and the predicted noise, then re-noise that estimate to the previous, less-noisy timestep. The probe uses the true noise, so <code>x0_hat</code> should come out exactly 2.0.</div>
</details>

In [ ]:
def ddim_step(x_t, predicted_noise, alpha_bar_t, alpha_bar_prev):
    """One deterministic DDIM reverse step from timestep t to a previous timestep."""
    # STUDENT TASK 3:
    #   x0_hat = (x_t - sqrt(1 - alpha_bar_t) * predicted_noise) / sqrt(alpha_bar_t)
    #   return sqrt(alpha_bar_prev) * x0_hat + sqrt(1 - alpha_bar_prev) * predicted_noise
    raise NotImplementedError("Return the less-noisy x at the previous timestep.")

In [ ]:
ddim_x_t = torch.tensor([1.6])
ddim_noise = torch.tensor([0.5])
ddim_alpha_bar_t = torch.tensor(0.36)
ddim_alpha_bar_prev = torch.tensor(0.64)
ddim_x_prev = ddim_step(ddim_x_t, ddim_noise, ddim_alpha_bar_t, ddim_alpha_bar_prev)
ddim_probe_value = round(ddim_x_prev.item(), 4)
ddim_probe_contract = abs(ddim_probe_value - 1.9) < 1e-4
assert ddim_probe_contract, "With the true noise, x0_hat recovers 2.0 and the step returns 1.9."
print("Task 3 checks passed. DDIM reverse value:", ddim_probe_value)

## Provided: VAE rails

Every VAE run uses your `vae_objective`, the same seed, and a disjoint validation
split. The histories keep the positive objective loss, reconstruction term, and KL
term separate so you can tell which term changed. The encoder/decoder width comes
from the design you choose in task 4.

In [ ]:
class VAE(nn.Module):
    def __init__(self, hidden_dim, latent_dim, input_dim=784):
        super().__init__()
        self.enc_fc1 = nn.Linear(input_dim, hidden_dim)
        self.enc_mu = nn.Linear(hidden_dim, latent_dim)
        self.enc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.dec_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = torch.relu(self.enc_fc1(x))
        return self.enc_mu(h), self.enc_logvar(h)

    def decode(self, z):
        return self.dec_out(torch.relu(self.dec_fc1(z)))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decode(z), mu, logvar


def build_vae(hidden_dim, latent_dim):
    return VAE(hidden_dim, latent_dim)


@dataclass(frozen=True)
class VAEExperiment:
    hidden_dim: int = 64
    latent_dim: int = 16
    beta: float = 1.0
    learning_rate: float = 1e-3
    batch_size: int = DEFAULT_BATCH_SIZE
    epochs: int = EPOCHS
    seed: int = SEED


def changed_config_fields(reference, candidate):
    reference_fields = asdict(reference)
    candidate_fields = asdict(candidate)
    return {
        name for name in reference_fields
        if reference_fields[name] != candidate_fields[name]
    }


def history_is_complete(record):
    history = record["history"]
    epochs = record["config"]["epochs"]
    return all(
        len(history[metric]) == epochs
        for metric in (
            "train_loss", "train_reconstruction", "train_kl",
            "validation_loss", "validation_reconstruction", "validation_kl",
        )
    )


@torch.no_grad()
def evaluate_vae(model, loader, beta):
    model.eval()
    total_loss = total_reconstruction = total_kl = examples = 0.0
    for images, _ in loader:
        x = images.view(images.shape[0], -1).to(DEVICE)
        reconstructed, mu, logvar = model(x)
        loss, reconstruction, kl = vae_objective(reconstructed, x, mu, logvar, beta)
        batch_size = x.shape[0]
        total_loss += loss.item() * batch_size
        total_reconstruction += reconstruction.item() * batch_size
        total_kl += kl.item() * batch_size
        examples += batch_size
    return (
        total_loss / examples,
        total_reconstruction / examples,
        total_kl / examples,
    )


def run_vae(label, config):
    # Train one VAE configuration with your vae_objective and return its config and full
    # history, including held-out objective, reconstruction, and KL terms each epoch.
    reset_seed(config.seed)
    model = build_vae(config.hidden_dim, config.latent_dim).to(DEVICE)
    fit_loader = seeded_loader(
        train_set, batch_size=config.batch_size, shuffle=True, seed=config.seed
    )
    train_eval_loader = seeded_loader(
        train_set, batch_size=config.batch_size, shuffle=False, seed=config.seed
    )
    validation_loader = seeded_loader(
        validation_set, batch_size=config.batch_size, shuffle=False, seed=config.seed
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    history = {
        "train_loss": [], "train_reconstruction": [], "train_kl": [],
        "validation_loss": [], "validation_reconstruction": [], "validation_kl": [],
    }
    for epoch in range(1, config.epochs + 1):
        model.train()
        for images, _ in fit_loader:
            x = images.view(images.shape[0], -1).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            reconstructed, mu, logvar = model(x)
            loss, _, _ = vae_objective(reconstructed, x, mu, logvar, config.beta)
            loss.backward(); optimizer.step()
        train_loss, train_reconstruction, train_kl = evaluate_vae(
            model, train_eval_loader, config.beta
        )
        validation_loss, validation_reconstruction, validation_kl = evaluate_vae(
            model, validation_loader, config.beta
        )
        history["train_loss"].append(train_loss)
        history["train_reconstruction"].append(train_reconstruction)
        history["train_kl"].append(train_kl)
        history["validation_loss"].append(validation_loss)
        history["validation_reconstruction"].append(validation_reconstruction)
        history["validation_kl"].append(validation_kl)
        print(f"{label:>22} | epoch {epoch}/{config.epochs} | "
              f"held-out loss {validation_loss:.2f} | "
              f"reconstruction {validation_reconstruction:.2f} | KL {validation_kl:.2f}")
    record = {
        "label": label,
        "config": asdict(config),
        "history": history,
        "model": model.eval(),
    }
    return record


baseline_config = VAEExperiment()
baseline_result = run_vae("vae baseline", baseline_config)
anchor_config = replace(baseline_config, latent_dim=2)
anchor_result = run_vae("small-latent anchor", anchor_config)

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 4</div>
<div style="font-weight:700;font-size:1.2rem;">Design a VAE</div>
</div>

Choose `hidden_dim` so the VAE has **100,000 to 800,000** trainable parameters. The
VAE has `1570 * hidden_dim + 3 * 16 * hidden_dim + 2 * 16 + 784` parameters at the
default latent dimension of 16.

In [ ]:
# STUDENT TASK 4: pick a positive hidden_dim inside the budget.
hidden_dim = 0

if hidden_dim <= 0:
    raise ValueError("Choose a positive hidden_dim.")
learner_config = replace(baseline_config, hidden_dim=hidden_dim)
learner_vae = build_vae(learner_config.hidden_dim, learner_config.latent_dim)
learner_vae_parameter_count = trainable_parameter_count(learner_vae)
learner_parameter_count = learner_vae_parameter_count
if not (100_000 <= learner_parameter_count <= 800_000):
    raise ValueError("Choose hidden_dim so the VAE has 100,000-800,000 parameters.")
print(f"learner VAE parameters: {learner_parameter_count:,}")

In [ ]:
learner_result = run_vae("learner vae design", learner_config)
for record in (baseline_result, anchor_result, learner_result):
    epochs = np.arange(1, len(record["history"]["validation_loss"]) + 1)
    plt.plot(epochs, record["history"]["validation_loss"], marker="o", label=record["label"])
plt.xlabel("epoch"); plt.ylabel("held-out positive objective loss")
plt.legend(); plt.tight_layout(); plt.show()

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 5</div>
<div style="font-weight:700;font-size:1.2rem;">Choose one KL-weight setting</div>
</div>

Beta sets the reconstruction-regularization tradeoff. Before running the treatment,
choose exactly one beta change relative to the baseline and record directional
predictions for the held-out reconstruction term and KL. Use `lower`, `higher`, or
`similar` for each prediction. The one-factor contract permits only beta to change.

In [ ]:
# STUDENT TASK 5: choose "low_beta", "high_beta", or "tiny_beta".
BETA_CHOICE = ""
PREDICT_RECONSTRUCTION = ""  # "lower", "higher", or "similar"
PREDICT_KL = ""              # "lower", "higher", or "similar"

BETA_OVERRIDES = {
    "low_beta": {"beta": 0.25},
    "high_beta": {"beta": 4.0},
    "tiny_beta": {"beta": 0.1},
}
BETA_LABELS = {
    "low_beta": "chosen beta: 0.25",
    "high_beta": "chosen beta: 4.0",
    "tiny_beta": "chosen beta: 0.1",
}
if BETA_CHOICE not in BETA_OVERRIDES:
    raise ValueError("Choose low_beta, high_beta, or tiny_beta.")
if PREDICT_RECONSTRUCTION not in {"lower", "higher", "similar"}:
    raise ValueError("Record a reconstruction prediction: lower, higher, or similar.")
if PREDICT_KL not in {"lower", "higher", "similar"}:
    raise ValueError("Record a KL prediction: lower, higher, or similar.")
treatment_config = replace(baseline_config, **BETA_OVERRIDES[BETA_CHOICE])
assert changed_config_fields(baseline_config, treatment_config) == {"beta"}

In [ ]:
treatment_result = run_vae(BETA_LABELS[BETA_CHOICE], treatment_config)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for record in (baseline_result, treatment_result):
    epochs = np.arange(1, len(record["history"]["validation_loss"]) + 1)
    for axis, metric, title in zip(
        axes,
        ("validation_loss", "validation_reconstruction", "validation_kl"),
        ("positive objective loss", "reconstruction term", "KL term"),
    ):
        axis.plot(epochs, record["history"][metric], marker="o", label=record["label"])
        axis.set_xlabel("epoch"); axis.set_title(title)
axes[0].set_ylabel("held-out mean")
axes[-1].legend(loc="best", fontsize=8)
plt.tight_layout(); plt.show()

vae_metric_table = {
    record["label"]: {
        metric: record["history"][metric][-1]
        for metric in ("validation_loss", "validation_reconstruction", "validation_kl")
    }
    for record in (baseline_result, treatment_result)
}
print("Final held-out evidence (lower loss/reconstruction is better; KL is descriptive):")
for label, metrics in vae_metric_table.items():
    print(label, {name: round(value, 3) for name, value in metrics.items()})

In [ ]:
@torch.no_grad()
def show_vae_evidence(baseline_record, treatment_record, n=8):
    # Posterior-mean reconstructions remove sampling noise. Both models decode the
    # same fixed prior draws, so differences are attributable to the beta treatment.
    validation_loader = seeded_loader(
        validation_set, batch_size=n, shuffle=False, seed=SEED
    )
    images, _ = next(iter(validation_loader))
    x = images[:n].view(n, -1).to(DEVICE)
    generator = torch.Generator(device="cpu").manual_seed(SEED + 17)
    fixed_z = torch.randn(n, baseline_config.latent_dim, generator=generator).to(DEVICE)

    rows = [("held-out input", x)]
    for record in (baseline_record, treatment_record):
        model = record["model"]
        mu, _ = model.encode(x)
        rows.append((f'{record["label"]}: reconstruction', model.decode(mu)))
    for record in (baseline_record, treatment_record):
        rows.append((f'{record["label"]}: prior sample', record["model"].decode(fixed_z)))

    fig, axes = plt.subplots(len(rows), n, figsize=(12, 7))
    for row_index, (label, values) in enumerate(rows):
        for column in range(n):
            axes[row_index, column].imshow(
                values[column].view(28, 28).detach().cpu().clamp(0, 1), cmap="gray"
            )
            axes[row_index, column].axis("off")
        axes[row_index, 0].set_ylabel(label, fontsize=8)
    plt.tight_layout(); plt.show()


show_vae_evidence(baseline_result, treatment_result)

## Provided: diffusion evidence arc

This arc trains a small denoiser whose target is built by your `q_sample`, then
generates fixed-seed samples with your `ddim_step` at two valid sampling budgets.
The schedule has 100 distinct trained levels, includes the clean endpoint explicitly,
and ends close enough to Gaussian noise to justify a pure-noise initial state. A
held-out diagnostic separates early, middle, and late noise levels.

In [ ]:
DIFFUSION_STEPS = 100


def cosine_beta_schedule(steps, offset=0.008):
    # alpha_bar[0] is the clean endpoint. Clipping the final beta avoids division
    # by exactly zero while leaving the terminal signal weight below 0.001.
    grid = torch.arange(steps + 1, dtype=torch.float64)
    target_alpha_bar = torch.cos(
        ((grid / steps + offset) / (1 + offset)) * math.pi / 2
    ).pow(2)
    target_alpha_bar = target_alpha_bar / target_alpha_bar[0]
    betas = 1 - target_alpha_bar[1:] / target_alpha_bar[:-1]
    return betas.clamp(1e-4, 0.999).float()


diffusion_betas = cosine_beta_schedule(DIFFUSION_STEPS).to(DEVICE)
ALPHA_BAR = torch.cat([
    torch.ones(1, device=DEVICE),
    torch.cumprod(1.0 - diffusion_betas, dim=0),
])
terminal_signal_weight = ALPHA_BAR[-1].sqrt().item()
assert len(ALPHA_BAR) == DIFFUSION_STEPS + 1
assert terminal_signal_weight < 0.001
print(
    f"clean alpha_bar={ALPHA_BAR[0].item():.1f}; "
    f"terminal alpha_bar={ALPHA_BAR[-1].item():.2e}; "
    f"terminal signal weight={terminal_signal_weight:.2e}"
)


class Denoiser(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784 + 1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 784),
        )

    def forward(self, x_t, t):
        t_feature = (t.float() / DIFFUSION_STEPS).view(-1, 1)
        return self.net(torch.cat([x_t, t_feature], dim=1))


def train_diffusion(seed=SEED, epochs=EPOCHS):
    # Train the denoiser to predict the noise added by your q_sample: draw a timestep and
    # noise, form the noised x_t, and regress the model's output onto that noise.
    reset_seed(seed)
    model = Denoiser().to(DEVICE)
    loader = seeded_loader(train_set, batch_size=DEFAULT_BATCH_SIZE, shuffle=True, seed=seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs + 1):
        model.train()
        last = 0.0
        for images, _ in loader:
            x_0 = images.view(images.shape[0], -1).to(DEVICE)
            t = torch.randint(1, DIFFUSION_STEPS + 1, (x_0.shape[0],), device=DEVICE)
            noise = torch.randn_like(x_0)
            alpha_bar = ALPHA_BAR[t].view(-1, 1)
            x_t = q_sample(x_0, alpha_bar, noise)  # your forward step builds the target
            loss = ((model(x_t, t) - noise) ** 2).mean()
            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            last = loss.item()
        print(f"diffusion epoch {epoch}/{epochs} | noise MSE {last:.4f}")
    return model.eval()


@torch.no_grad()
def sampling_schedule(sampling_steps):
    if not 2 <= sampling_steps <= DIFFUSION_STEPS:
        raise ValueError(f"Choose 2 through {DIFFUSION_STEPS} distinct sampling levels.")
    schedule = torch.linspace(
        DIFFUSION_STEPS, 1, sampling_steps, device=DEVICE
    ).round().long()
    assert len(torch.unique(schedule)) == sampling_steps
    assert schedule[0].item() == DIFFUSION_STEPS and schedule[-1].item() == 1
    return schedule


@torch.no_grad()
def ddim_sample(model, n_samples, sampling_steps):
    # Both budgets begin from the same seeded noise. Each denoiser call uses one
    # distinct trained level; alpha_bar[0]=1 supplies the final clean endpoint.
    reset_seed(SEED)
    x = torch.randn(n_samples, 784, device=DEVICE)
    schedule = sampling_schedule(sampling_steps)
    for i in range(len(schedule)):
        t = schedule[i]
        t_prev = schedule[i + 1] if i + 1 < len(schedule) else torch.tensor(0, device=DEVICE)
        predicted = model(x, t.repeat(n_samples))
        x = ddim_step(x, predicted, ALPHA_BAR[t], ALPHA_BAR[t_prev])
    return x


@torch.no_grad()
def heldout_denoising_diagnostics(model, levels=(10, 50, 100), n=128):
    loader = seeded_loader(validation_set, batch_size=n, shuffle=False, seed=SEED)
    images, _ = next(iter(loader))
    x_0 = images[:n].view(-1, 784).to(DEVICE)
    reset_seed(SEED + 23)
    fixed_noise = torch.randn_like(x_0)
    evidence = {}
    for level in levels:
        t = torch.full((x_0.shape[0],), level, device=DEVICE, dtype=torch.long)
        x_t = q_sample(x_0, ALPHA_BAR[level].view(1, 1), fixed_noise)
        evidence[level] = ((model(x_t, t) - fixed_noise) ** 2).mean().item()
    return evidence


denoiser = train_diffusion()
denoising_evidence = heldout_denoising_diagnostics(denoiser)
print("Held-out noise-prediction MSE by timestep:", {
    level: round(value, 4) for level, value in denoising_evidence.items()
})
samples_25 = ddim_sample(denoiser, 8, sampling_steps=25)
samples_100 = ddim_sample(denoiser, 8, sampling_steps=100)
assert len(sampling_schedule(25)) == len(torch.unique(sampling_schedule(25))) == 25
assert len(sampling_schedule(100)) == len(torch.unique(sampling_schedule(100))) == 100
diffusion_schedule_contract = int(
    ALPHA_BAR[0].item() == 1.0
    and terminal_signal_weight < 0.001
    and len(torch.unique(sampling_schedule(25))) == 25
    and len(torch.unique(sampling_schedule(100))) == 100
)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for column in range(8):
    axes[0, column].imshow(samples_25[column].view(28, 28).cpu(), cmap="gray")
    axes[1, column].imshow(samples_100[column].view(28, 28).cpu(), cmap="gray")
    axes[0, column].axis("off"); axes[1, column].axis("off")
axes[0, 0].set_ylabel("25 steps"); axes[1, 0].set_ylabel("100 steps")
plt.tight_layout(); plt.show()
print("Fixed-seed samples: 25 distinct calls versus 100 distinct calls.")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 6</div>
<div style="font-weight:700;font-size:1.2rem;">Write a claim-evidence-caveat record</div>
</div>

Use your decomposed held-out table and fixed-seed image grid. Write one bounded claim,
cite at least one numeric comparison and one visual observation, name a caveat, and
propose a next experiment that changes one factor. Your claim need not favor the
treatment. The final contract checks that the record is complete, not whether you
obtained a prescribed stochastic result.

In [ ]:
# STUDENT TASK 6: replace each empty string with your own concise record.
CLAIM = ""
EVIDENCE = ""
CAVEAT = ""
NEXT_EXPERIMENT = ""

learner_record = {
    "prediction_reconstruction": PREDICT_RECONSTRUCTION,
    "prediction_kl": PREDICT_KL,
    "claim": CLAIM.strip(),
    "evidence": EVIDENCE.strip(),
    "caveat": CAVEAT.strip(),
    "next_experiment": NEXT_EXPERIMENT.strip(),
}
if any(len(learner_record[field]) < 12 for field in (
    "claim", "evidence", "caveat", "next_experiment"
)):
    raise ValueError("Complete each record field with a specific statement (12+ characters).")
learner_record_contract = 1
print("Experiment record captured:", learner_record)

## Pause and reflect (ungraded)

Do held-out denoising MSE and visual sample quality tell the same story across noise
levels? Did 100 distinct calls look meaningfully better than 25 for this checkpoint?
Neither outcome is guaranteed. Name an alternative explanation before generalizing.

## Report values

Run the cell below after completing all six tasks and all four VAE runs. R1-R3
are the readiness values entered in Gate 1. R4 and R5 form the experiment-record
submission in Gate 2. R4 is **your** valid VAE design's parameter count in the
100,000-800,000 range. R5 verifies the split, one-factor comparisons, decomposed
histories, diffusion schedule, and completed evidence record. Gates 3 and 4 assess
claim-evidence-caveat reasoning and next-experiment design.

In [ ]:
run_config_pairs = (
    (baseline_result, baseline_config),
    (anchor_result, anchor_config),
    (learner_result, learner_config),
    (treatment_result, treatment_config),
)

workflow_contract = int(
    set(train_indices).isdisjoint(validation_indices)
    and len({record["label"] for record, _ in run_config_pairs}) == 4
    and all(record["config"] == asdict(config) for record, config in run_config_pairs)
    and all(history_is_complete(record) for record, _ in run_config_pairs)
    and changed_config_fields(baseline_config, anchor_config) == {"latent_dim"}
    and changed_config_fields(baseline_config, learner_config) == {"hidden_dim"}
    and changed_config_fields(baseline_config, treatment_config) == {"beta"}
    and learner_parameter_count == learner_vae_parameter_count
    and 100_000 <= learner_parameter_count <= 800_000
    and diffusion_schedule_contract == 1
    and learner_record_contract == 1
)

probe_kl = round(kl_probe_value, 5) if kl_probe_contract else -1.0
probe_forward = round(qsample_probe_value, 4) if qsample_probe_contract else -1.0
probe_reverse = round(ddim_probe_value, 4) if ddim_probe_contract else -1.0

report_values = {
    "R1: analytic VAE KL on the fixed probe": probe_kl,
    "R2: forward-diffusion noised value on the fixed probe": probe_forward,
    "R3: DDIM reverse-step value on the fixed probe": probe_reverse,
    "R4: learner-designed VAE parameter count": learner_parameter_count,
    "R5: experimental workflow contract": workflow_contract,
}
assert abs(probe_kl - 1.30685) < 1e-4
assert abs(probe_forward - 1.6) < 1e-4
assert abs(probe_reverse - 1.9) < 1e-4
assert 100_000 <= learner_parameter_count <= 800_000
assert workflow_contract == 1

print("LAB 5 REPORT VALUES")
for label, value in report_values.items():
    print(f"{label}: {value}")